In [5]:
import os
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
import kagglehub

tushar5harma_path    = kagglehub.dataset_download('tushar5harma/plant-village-dataset-updated')
shubham2703_path     = kagglehub.dataset_download('shubham2703/five-crop-diseases-dataset')
zunorain_path        = kagglehub.dataset_download('zunorain/pea-plant-dataset')
mexwell_path         = kagglehub.dataset_download('mexwell/crop-diseases-classification')
ayushbpanchal_path   = kagglehub.dataset_download('ayushbpanchal/cashew-image-dataset')
tahmidmir_path       = kagglehub.dataset_download('tahmidmir/pumpkin-leaf-diseases-dataset-from-bangladesh')
binnassor89_path     = kagglehub.dataset_download('binnassor89/melon-macronutrient-deficiency-dataset')
nirmalsankalana_path = kagglehub.dataset_download('nirmalsankalana/rocole-a-robusta-coffee-leaf-images-dataset')
yasaminborhani_path = kagglehub.dataset_download('yasaminborhani/wheatrustclassificationdataset')

print('Downloads complete.')

Downloads complete.


# Hyperparameters

In [6]:
PHASE_B_CHECKPOINT = "/kaggle/input/models/vitthaljauhari/resnet50-phase2-pth/pytorch/default/1/phase2_best.pth"
RUNS_DIR           = "/kaggle/working/runs"
os.makedirs(RUNS_DIR, exist_ok=True)

BASE        = "/kaggle/input/datasets"   # rglob root per dataset
IMAGE_EXTS  = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

BATCH_SIZE          = 128
NUM_WORKERS         = 4
SEED                = 42
N_EPOCHS            = 15
IMG_SIZE            = 224
EARLY_STOP_PATIENCE = 3

LR_MAP = {
    "conv1_bn1": 1e-5,
    "layer1":    1e-5,
    "layer2":    2e-5,
    "layer3":    3e-5,
    "layer4":    5e-5,
    "fc":        5e-5,
}

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | GPUs: {torch.cuda.device_count()}")

Device: cuda | GPUs: 2


# DataClasses

In [7]:
dataset_label_maps = {

    # tushar5harma: keyed by (crop_dir, disease_dir) — 3-level: Crop/Split/Disease
    "tushar5harma": {
        ("Apple",        "Apple Scab")             : "Apple__Apple_Scab",
        ("Apple",        "Black Rot")              : "Apple__Black_Rot",
        ("Apple",        "Cedar Apple Rust")       : "Apple__Cedar_Apple_Rust",
        ("Apple",        "Healthy")                : "Apple__Healthy",
        ("Bell Pepper",  "Bacterial Spot")         : "Bell_Pepper__Bacterial_Spot",
        ("Bell Pepper",  "Healthy")                : "Bell_Pepper__Healthy",
        ("Cherry",       "Healthy")                : "Cherry__Healthy",
        ("Cherry",       "Powdery Mildew")         : "Cherry__Powdery_Mildew",
        ("Corn (Maize)", "Cercospora Leaf Spot")   : "Corn__Gray_Leaf_Spot",
        ("Corn (Maize)", "Common Rust")            : "Corn__Common_Rust",
        ("Corn (Maize)", "Healthy")                : "Corn__Healthy",
        ("Corn (Maize)", "Northern Leaf Blight")   : "Corn__Northern_Leaf_Blight",
        ("Grape",        "Black Rot")              : "Grape__Black_Rot",
        ("Grape",        "Esca (Black Measles)")   : "Grape__Black_Measles",
        ("Grape",        "Healthy")                : "Grape__Healthy",
        ("Grape",        "Leaf Blight")            : "Grape__Leaf_Blight",
        ("Peach",        "Bacterial Spot")         : "Peach__Bacterial_Spot",
        ("Peach",        "Healthy")                : "Peach__Healthy",
        ("Potato",       "Early Blight")           : "Potato__Early_Blight",
        ("Potato",       "Healthy")                : "Potato__Healthy",
        ("Potato",       "Late Blight")            : "Potato__Late_Blight",
        ("Strawberry",   "Healthy")                : "Strawberry__Healthy",
        ("Strawberry",   "Leaf Scorch")            : "Strawberry__Leaf_Scorch",
        ("Tomato",       "Bacterial Spot")         : "Tomato__Bacterial_Spot",
        ("Tomato",       "Early Blight")           : "Tomato__Early_Blight",
        ("Tomato",       "Healthy")                : "Tomato__Healthy",
        ("Tomato",       "Late Blight")            : "Tomato__Late_Blight",
        ("Tomato",       "Septoria Leaf Spot")     : "Tomato__Septoria_Leaf_Spot",
        ("Tomato",       "Yellow Leaf Curl Virus") : "Tomato__Yellow_Leaf_Curl_Virus",
    },

    # shubham2703: structure is Crop Diseases/Crop___Disease/{Crop}/{Class}/*.jpg
    # rglob parent.name gives the leaf class folder name e.g. "Corn___Common_Rust"
    "shubham2703": {
        "Corn___Common_Rust"          : "Corn__Common_Rust",
        "Corn___Gray_Leaf_Spot"       : "Corn__Gray_Leaf_Spot",
        "Corn___Healthy"              : "Corn__Healthy",
        "Corn___Northern_Leaf_Blight" : "Corn__Northern_Leaf_Blight",
        "Potato___Early_Blight"       : "Potato__Early_Blight",
        "Potato___Healthy"            : "Potato__Healthy",
        "Potato___Late_Blight"        : "Potato__Late_Blight",
        "Rice___Brown_Spot"           : "Rice__Brown_Spot",
        "Rice___Healthy"              : "Rice__Healthy",
        "Rice___Leaf_Blast"           : "Rice__Leaf_Blast",
        "Rice___Neck_Blast"           : "Rice__Neck_Blast",
        "Wheat___Brown_Rust"          : "Wheat__Brown_Rust",
        "Wheat___Healthy"             : "Wheat__Healthy",
        "Wheat___Yellow_Rust"         : "Wheat__Yellow_Rust",
        "Bacterial Blight"            : "Rice__Bacterial_Blight",
        "Red Rot"                     : "Sugarcane__Red_Rot",
    },

    # nirmalsankalana: only coffee classes present on disk
    "nirmalsankalana": {
        "coffee___healthy"         : "Coffee__Healthy",
        "coffee___red_spider_mite" : "Coffee__Red_Spider_Mite",
        "coffee___rust"            : "Coffee__Rust",
        "BlackMeasles"             : "Grape__Black_Measles",
        "BlackRot"                 : "Grape__Black_Rot",
        "HealthyGrapes"            : "Grape__Healthy",
        "LeafBlight"               : "Grape__Leaf_Blight",
    },

    # binnassor89: actual folder names confirmed from disk
    "binnassor89": {
        "calcium"   : "Melon__Calcium_Deficiency",
        "health"    : "Melon__Healthy",
        "nitrogen"  : "Melon__Nitrogen_Deficiency",
        "potasium"  : "Melon__Potassium_Deficiency",   # sic — matches disk
    },

    # zunorain: actual folder names confirmed from disk
    "zunorain": {
        "DOWNY_MILDEW_LEAF"  : "Pea__Downy_Mildew",
        "FRESH_LEAF"         : "Pea__Healthy",
        "LEAFMINNER_LEAF"    : "Pea__Leaf_Miner",
        "POWDER_MILDEW_LEAF" : "Pea__Powdery_Mildew",
    },

    # ayushbpanchal: train_set + test_set both present; rglob picks both
    "ayushbpanchal": {
        "anthracnose" : "Cashew__Anthracnose",
        "gumosis"     : "Cashew__Gummosis",
        "healthy"     : "Cashew__Healthy",
        "leaf miner"  : "Cashew__Leaf_Miner",
        "red rust"    : "Cashew__Red_Rust",
    },

    # tahmidmir: Original Dataset/{Class}/*.jpg — rglob parent.name = class folder
    "tahmidmir": {
        "Bacterial Leaf Spot" : "Pumpkin__Bacterial_Leaf_Spot",
        "Downy Mildew"        : "Pumpkin__Downy_Mildew",
        "Healthy Leaf"        : "Pumpkin__Healthy",
        "Mosaic Disease"      : "Pumpkin__Mosaic_Disease",
        "Powdery_Mildew"      : "Pumpkin__Powdery_Mildew",
    },

    # mexwell: CSV-based — label resolved via label_num_to_disease_map.json
    "mexwell": {
        "Cassava Bacterial Blight (CBB)"      : "Cassava__Bacterial_Blight",
        "Cassava Brown Streak Disease (CBSD)" : "Cassava__Brown_Streak_Disease",
        "Cassava Green Mottle (CGM)"          : "Cassava__Green_Mottle",
        "Cassava Mosaic Disease (CMD)"        : "Cassava__Mosaic_Disease",
        "Healthy"                             : "Cassava__Healthy",
    },
    "yasaminborhani": {
        "Brown_rust"  : "Wheat__Brown_Rust",
        "Healthy"     : "Wheat__Healthy",
        "Yellow_rust" : "Wheat__Yellow_Rust",
    },
}

# Build global class_to_idx  (must match Phase B exactly — same label maps)
all_canonical = sorted(set(
    v for mapping in dataset_label_maps.values()
    for v in mapping.values()
    if v is not None
))
class_to_idx = {cls: idx for idx, cls in enumerate(all_canonical)}
N_CLASSES    = len(class_to_idx)
print(f"Total canonical classes: {N_CLASSES}")

Total canonical classes: 64


In [8]:
def build_samples_folder(slug: str) -> list:
    """
    rglob over the dataset root; match on img_path.parent.name.
    Handles arbitrary nesting (shubham2703's 4-level, ayushbpanchal's train/test, etc.)
    """
    label_map = dataset_label_maps[slug]
    root      = Path(BASE) / slug
    samples, skipped = [], 0
    for img_path in root.rglob("*"):
        if img_path.suffix.lower() not in IMAGE_EXTS:
            continue
        canonical = label_map.get(img_path.parent.name)
        if canonical is None:
            skipped += 1
            continue
        samples.append((str(img_path), class_to_idx[canonical]))
    print(f"  [{slug:<20}] {len(samples):>6} samples  ({skipped} skipped)")
    return samples


def build_samples_tushar() -> list:
    """
    Structure: {Crop}/{Split}/{Disease}/*.jpg
    Key = (crop_dir.name, disease_dir.name)
    """
    label_map = dataset_label_maps["tushar5harma"]
    root      = Path(BASE) / "tushar5harma"
    samples, skipped = [], 0
    for img_path in root.rglob("*"):
        if img_path.suffix.lower() not in IMAGE_EXTS:
            continue
        # img_path.parents: [disease_dir, split_dir, crop_dir, root, ...]
        disease_dir = img_path.parent
        split_dir   = disease_dir.parent
        crop_dir    = split_dir.parent
        key = (crop_dir.name, disease_dir.name)
        canonical = label_map.get(key)
        if canonical is None:
            skipped += 1
            continue
        samples.append((str(img_path), class_to_idx[canonical]))
    print(f"  [{'tushar5harma':<20}] {len(samples):>6} samples  ({skipped} skipped)")
    return samples


def build_samples_mexwell() -> list:
    """
    CSV-based. Label resolved via label_num_to_disease_map.json (int key → disease name).
    """
    data_root = Path(BASE) / "mexwell" / "crop-diseases-classification" / "Data"
    img_dir   = data_root / "train_images"
    json_path = data_root / "label_num_to_disease_map.json"
    csv_path  = data_root / "train.csv"

    with open(json_path) as f:
        num_to_name = json.load(f)   # {"0": "Cassava Bacterial Blight (CBB)", ...}

    df        = pd.read_csv(csv_path)
    label_map = dataset_label_maps["mexwell"]
    samples, skipped = [], 0

    for _, row in df.iterrows():
        img_path     = img_dir / row["image_id"]
        if not img_path.exists():
            skipped += 1
            continue
        disease_name = num_to_name[str(row["label"])]
        canonical    = label_map.get(disease_name)
        if canonical is None:
            skipped += 1
            continue
        samples.append((str(img_path), class_to_idx[canonical]))

    print(f"  [{'mexwell':<20}] {len(samples):>6} samples  ({skipped} skipped)")
    return samples


def build_all_samples():
    print("Loading datasets...")
    folder_slugs = ["shubham2703", "nirmalsankalana", "binnassor89",
                    "zunorain", "ayushbpanchal", "tahmidmir", "yasaminborhani"]
    all_samples = []
    for slug in folder_slugs:
        all_samples += build_samples_folder(slug)
    all_samples += build_samples_mexwell()
    all_samples += build_samples_tushar()
    print(f"\nTotal: {len(all_samples):,} samples | "
          f"{len(set(lbl for _, lbl in all_samples))} unique class indices")
    return all_samples

# Transforms

In [9]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# DataSet  Builder

In [10]:
class CropDiseaseDataset(Dataset):
    """samples: list of (path_str, class_idx_int) — same format as Phase B."""
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

# DataLoader

In [11]:
def build_dataloaders(all_samples):
    # Stratify on label index
    labels     = [lbl for _, lbl in all_samples]
    train_s, val_s = train_test_split(
        all_samples, test_size=0.15, random_state=SEED, stratify=labels
    )

    train_ds = CropDiseaseDataset(train_s, transform=train_transform)
    val_ds   = CropDiseaseDataset(val_s,   transform=val_transform)

    # WeightedRandomSampler
    train_labels  = [lbl for _, lbl in train_s]
    class_counts  = np.bincount(train_labels, minlength=N_CLASSES)
    class_weights = 1.0 / (class_counts + 1e-6)
    sample_w      = [class_weights[l] for l in train_labels]
    sampler       = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)

    kw = dict(num_workers=NUM_WORKERS, prefetch_factor=4,
              persistent_workers=True, pin_memory=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, **kw)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,   **kw)

    print(f"Train: {len(train_ds):,}  |  Val: {len(val_ds):,}")
    return train_loader, val_loader

# Model Instantiation

In [12]:
def load_phase_b_model(path):
    ckpt = torch.load(path, map_location="cpu")
    assert ckpt["n_classes"] == N_CLASSES, \
        f"Checkpoint n_classes={ckpt['n_classes']} but label maps give {N_CLASSES}. Mismatch."

    model = models.resnet50(weights=None)
    model.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(2048, N_CLASSES))

    state = ckpt["model_state_dict"]
    if any(k.startswith("module.") for k in state):
        state = {k.replace("module.", "", 1): v for k, v in state.items()}
    model.load_state_dict(state)
    print(f"Loaded Phase B | epoch={ckpt['epoch']} | best_acc={ckpt['best_acc']:.4f}")
    return model, ckpt


def build_optimizer(model):
    for p in model.parameters():
        p.requires_grad = True
    base = model.module if hasattr(model, "module") else model
    return optim.Adam([
        {"params": list(base.conv1.parameters()) + list(base.bn1.parameters()),
         "lr": LR_MAP["conv1_bn1"], "name": "conv1_bn1"},
        {"params": base.layer1.parameters(), "lr": LR_MAP["layer1"], "name": "layer1"},
        {"params": base.layer2.parameters(), "lr": LR_MAP["layer2"], "name": "layer2"},
        {"params": base.layer3.parameters(), "lr": LR_MAP["layer3"], "name": "layer3"},
        {"params": base.layer4.parameters(), "lr": LR_MAP["layer4"], "name": "layer4"},
        {"params": base.fc.parameters(),     "lr": LR_MAP["fc"],     "name": "fc"},
    ])


# Training

In [9]:
def accuracy_topk(output, target, topk=(1, 5)):
    with torch.no_grad():
        maxk = max(topk)
        bs   = target.size(0)
        _, pred = output.topk(maxk, dim=1, largest=True, sorted=True)
        correct = pred.t().eq(target.view(1, -1).expand_as(pred.t()))
        return [correct[:k].reshape(-1).float().sum().mul_(100.0 / bs).item() for k in topk]


def train_one_epoch(model, loader, criterion, optimizer, epoch):
    model.train()
    loss_sum = top1_sum = top5_sum = 0.0
    t0 = time.time()
    for i, (imgs, labels) in enumerate(loader):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        top1, top5 = accuracy_topk(out, labels)
        loss_sum += loss.item(); top1_sum += top1; top5_sum += top5
        if (i + 1) % 50 == 0:
            print(f"  [E{epoch} {i+1}/{len(loader)}] "
                  f"loss={loss_sum/(i+1):.4f}  top1={top1_sum/(i+1):.2f}%  "
                  f"t={time.time()-t0:.0f}s")
    n = len(loader)
    return loss_sum / n, top1_sum / n, top5_sum / n


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    t1 = t5 = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        a1, a5 = accuracy_topk(out, labels)
        bs = labels.size(0)
        t1 += a1 * bs; t5 += a5 * bs; total += bs
    return t1 / total, t5 / total

# =============================================================================
# CHECKPOINT
# =============================================================================

def save_checkpoint(model, optimizer, scheduler, epoch, val_top1, tag,
                    history=None, no_improve=0):
    base = model.module if hasattr(model, "module") else model
    path = os.path.join(RUNS_DIR, f"phaseC_{tag}.pth")
    torch.save({
        "epoch":                epoch,
        "phase":                "C",
        "model_state_dict":     base.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_acc":             val_top1,
        "class_to_idx":         class_to_idx,
        "n_classes":            N_CLASSES,
        "history":              history or [],
        "no_improve":           no_improve,
    }, path)
    print(f"  → phaseC_{tag}.pth  (top1={val_top1:.4f})")

# =============================================================================
# MAIN
# =============================================================================

def main():
    RESUME_CHECKPOINT = "/kaggle/working/runs/phaseC_final.pth"

    # 1. Model
    model, ckpt = load_phase_b_model(PHASE_B_CHECKPOINT)
    if torch.cuda.device_count() > 1:
        print(f"DataParallel: {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)
    model = model.to(device)

    # 2. Data
    all_samples              = build_all_samples()
    train_loader, val_loader = build_dataloaders(all_samples)

    # 3. Optimizer + scheduler
    optimizer = build_optimizer(model)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS, eta_min=1e-7)
    criterion = nn.CrossEntropyLoss()

    # 4. Resume if checkpoint exists
    start_epoch = 0
    best_top1   = ckpt.get("best_acc", 0.0)
    no_improve  = 0
    history     = []

    if os.path.exists(RESUME_CHECKPOINT):
        print(f"\nResuming from {RESUME_CHECKPOINT}")
        resume = torch.load(RESUME_CHECKPOINT, map_location="cpu")

        # Load model weights
        state = resume["model_state_dict"]
        if any(k.startswith("module.") for k in state):
            state = {k.replace("module.", "", 1): v for k, v in state.items()}
        base = model.module if hasattr(model, "module") else model
        base.load_state_dict(state)

        optimizer.load_state_dict(resume["optimizer_state_dict"])
        scheduler.load_state_dict(resume["scheduler_state_dict"])

        start_epoch = resume["epoch"] + 1
        best_top1   = resume["best_acc"]
        no_improve  = resume.get("no_improve", 0)
        history     = resume.get("history", [])

        print(f"  Resumed | start_epoch={start_epoch} | best_acc={best_top1:.4f} | no_improve={no_improve}")
    else:
        print(f"\nNo resume checkpoint found — starting fresh from Phase B")

    print(f"\n{'='*60}")
    print(f"Phase C — Full Backbone Unfreeze")
    print(f"Baseline top-1: {best_top1:.4f} | Target: >0.91 | Patience: {EARLY_STOP_PATIENCE}")
    print(f"{'='*60}")

    for epoch in range(start_epoch, N_EPOCHS):
        lr_str = " | ".join(f"{g['name']}={g['lr']:.1e}" for g in optimizer.param_groups)
        print(f"\n--- Epoch {epoch}/{N_EPOCHS-1}  [{lr_str}] ---")

        tr_loss, tr_t1, tr_t5 = train_one_epoch(model, train_loader, criterion, optimizer, epoch)
        val_t1, val_t5         = evaluate(model, val_loader)
        scheduler.step()

        history.append({"epoch": epoch, "loss": round(tr_loss, 4),
                        "tr_t1": round(tr_t1, 4), "val_t1": round(val_t1, 4),
                        "val_t5": round(val_t5, 4)})

        flag = ""
        if val_t1 > best_top1:
            best_top1  = val_t1
            no_improve = 0
            save_checkpoint(model, optimizer, scheduler, epoch, val_top1=val_t1,
                            tag="best", history=history, no_improve=no_improve)
            flag = "  ← best"
        else:
            no_improve += 1

        # Always save final for resume
        save_checkpoint(model, optimizer, scheduler, epoch, val_top1=val_t1,
                        tag="final", history=history, no_improve=no_improve)

        print(f"Epoch {epoch:02d} | loss={tr_loss:.4f} | "
              f"train={tr_t1:.2f}% | val_t1={val_t1:.2f}% | val_t5={val_t5:.2f}%{flag}")

        if tr_t1 - val_t1 > 8.0:
            print(f"  ⚠ gap={tr_t1 - val_t1:.1f}pp")
        if val_t1 >= 91.0:
            print("✅ Target reached.")
            break
        if no_improve >= EARLY_STOP_PATIENCE:
            print(f"⏹ Early stop ({EARLY_STOP_PATIENCE} epochs no improvement).")
            break

    print(f"\n{'='*60}\nPhase C Summary")
    print(f"{'Ep':>3} | {'Loss':>7} | {'Train%':>7} | {'Val T1':>7} | {'Val T5':>7}")
    for r in history:
        print(f"{r['epoch']:>3} | {r['loss']:>7.4f} | {r['tr_t1']:>7.2f} | "
              f"{r['val_t1']:>7.2f} | {r['val_t5']:>7.2f}")
    print(f"\nBest Val Top-1: {best_top1:.2f}%")


if __name__ == "__main__":
    main()

Loaded Phase B | epoch=20 | best_acc=0.8072
DataParallel: 2 GPUs
Loading datasets...


KeyboardInterrupt: 

# Training for a few More Epochs

In [10]:
# ============================================================
# REBUILD MODEL + DATALOADERS + CONTINUE TRAINING
# ============================================================

EXTRA_EPOCHS = 8
CONTINUE_CHECKPOINT = "/kaggle/working/runs/phaseC_best.pth"

# ------------------------------------------------------------
# REBUILD MODEL
# ------------------------------------------------------------

model, _ = load_phase_b_model(PHASE_B_CHECKPOINT)

if torch.cuda.device_count() > 1:
    print(f"DataParallel: {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

model = model.to(device)

# ------------------------------------------------------------
# LOAD CONTINUATION CHECKPOINT
# ------------------------------------------------------------

resume = torch.load(CONTINUE_CHECKPOINT, map_location="cpu")

base = model.module if hasattr(model, "module") else model

state = resume["model_state_dict"]

if any(k.startswith("module.") for k in state):
    state = {k.replace("module.", "", 1): v for k, v in state.items()}

base.load_state_dict(state)

best_top1 = resume["best_acc"]

print(f"Loaded continuation checkpoint | best_acc={best_top1:.4f}")

# ------------------------------------------------------------
# REBUILD DATA
# ------------------------------------------------------------

all_samples = build_all_samples()
train_loader, val_loader = build_dataloaders(all_samples)

# ------------------------------------------------------------
# NEW SMALL LR RESTART
# ------------------------------------------------------------

NEW_LR_MAP = {
    "conv1_bn1": 5e-6,
    "layer1":    5e-6,
    "layer2":    1e-5,
    "layer3":    2e-5,
    "layer4":    3e-5,
    "fc":        3e-5,
}

optimizer = optim.Adam([
    {
        "params": list(base.conv1.parameters()) + list(base.bn1.parameters()),
        "lr": NEW_LR_MAP["conv1_bn1"],
        "name": "conv1_bn1"
    },
    {
        "params": base.layer1.parameters(),
        "lr": NEW_LR_MAP["layer1"],
        "name": "layer1"
    },
    {
        "params": base.layer2.parameters(),
        "lr": NEW_LR_MAP["layer2"],
        "name": "layer2"
    },
    {
        "params": base.layer3.parameters(),
        "lr": NEW_LR_MAP["layer3"],
        "name": "layer3"
    },
    {
        "params": base.layer4.parameters(),
        "lr": NEW_LR_MAP["layer4"],
        "name": "layer4"
    },
    {
        "params": base.fc.parameters(),
        "lr": NEW_LR_MAP["fc"],
        "name": "fc"
    },
])

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EXTRA_EPOCHS,
    eta_min=1e-7
)

criterion = nn.CrossEntropyLoss()

# ------------------------------------------------------------
# CONTINUE TRAINING
# ------------------------------------------------------------

for epoch in range(EXTRA_EPOCHS):

    lr_str = " | ".join(
        f"{g['name']}={g['lr']:.1e}"
        for g in optimizer.param_groups
    )

    print(f"\n--- Extra Epoch {epoch+1}/{EXTRA_EPOCHS} [{lr_str}] ---")

    tr_loss, tr_t1, tr_t5 = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        epoch
    )

    val_t1, val_t5 = evaluate(model, val_loader)

    scheduler.step()

    if val_t1 > best_top1:
        best_top1 = val_t1

        save_checkpoint(
            model,
            optimizer,
            scheduler,
            epoch,
            val_top1=val_t1,
            tag="extra_best"
        )

        print(f"✅ NEW BEST: {best_top1:.2f}%")

    print(
        f"Epoch {epoch:02d} | "
        f"loss={tr_loss:.4f} | "
        f"train={tr_t1:.2f}% | "
        f"val_t1={val_t1:.2f}% | "
        f"val_t5={val_t5:.2f}%"
    )

print(f"\nFinal Best Accuracy: {best_top1:.2f}%")

Loaded Phase B | epoch=20 | best_acc=0.8072
DataParallel: 2 GPUs
Loaded continuation checkpoint | best_acc=87.3096
Loading datasets...
  [shubham2703         ]  13224 samples  (100 skipped)
  [nirmalsankalana     ]   3160 samples  (0 skipped)
  [binnassor89         ]    200 samples  (0 skipped)
  [zunorain            ]   1432 samples  (0 skipped)
  [ayushbpanchal       ]  25811 samples  (0 skipped)
  [tahmidmir           ]   2000 samples  (0 skipped)
  [yasaminborhani      ]   3679 samples  (0 skipped)
  [mexwell             ]  17938 samples  (3459 skipped)
  [tushar5harma        ]  64727 samples  (2391 skipped)

Total: 132,171 samples | 64 unique class indices
Train: 112,345  |  Val: 19,826

--- Extra Epoch 1/8 [conv1_bn1=5.0e-06 | layer1=5.0e-06 | layer2=1.0e-05 | layer3=2.0e-05 | layer4=3.0e-05 | fc=3.0e-05] ---
  [E0 50/878] loss=0.3474  top1=86.84%  t=99s
  [E0 100/878] loss=0.3473  top1=86.80%  t=189s
  [E0 150/878] loss=0.3415  top1=87.08%  t=275s
  [E0 200/878] loss=0.3400  top

# Evaluation

In [13]:
# ── Eval Cell 1: Load checkpoint & build model ───────────────────────────────
EVAL_CHECKPOINT = "/kaggle/working/runs/phaseC_extra_best.pth"

ckpt = torch.load(EVAL_CHECKPOINT, map_location="cpu")

# Use class_to_idx from checkpoint — guaranteed to match training
ckpt_class_to_idx = ckpt["class_to_idx"]
ckpt_idx_to_class = {v: k for k, v in ckpt_class_to_idx.items()}
ckpt_n_classes    = ckpt["n_classes"]

assert ckpt_n_classes == N_CLASSES, \
    f"Checkpoint n_classes={ckpt_n_classes} vs label maps {N_CLASSES}"
print(f"Classes: {ckpt_n_classes} | Best acc (val): {ckpt['best_acc']:.4f}")

# Rebuild model
eval_model = models.resnet50(weights=None)
eval_model.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(2048, ckpt_n_classes))

state = ckpt["model_state_dict"]
if any(k.startswith("module.") for k in state):
    state = {k.replace("module.", "", 1): v for k, v in state.items()}
eval_model.load_state_dict(state)
eval_model = eval_model.to(device).eval()
print("Model loaded.")

Classes: 64 | Best acc (val): 87.9653
Model loaded.


In [14]:
# ── Eval Cell 2: Build test set ───────────────────────────────────────────────
from sklearn.model_selection import train_test_split

all_samples  = build_all_samples()
labels       = [lbl for _, lbl in all_samples]

_, test_samples = train_test_split(
    all_samples,
    test_size=0.15,
    random_state=SEED,
    stratify=labels,
)

test_ds     = CropDiseaseDataset(test_samples, transform=val_transform)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)
print(f"Test set: {len(test_ds):,} samples")

Loading datasets...
  [shubham2703         ]  13224 samples  (100 skipped)
  [nirmalsankalana     ]   3160 samples  (0 skipped)
  [binnassor89         ]    200 samples  (0 skipped)
  [zunorain            ]   1432 samples  (0 skipped)
  [ayushbpanchal       ]  25811 samples  (0 skipped)
  [tahmidmir           ]   2000 samples  (0 skipped)
  [yasaminborhani      ]   3679 samples  (0 skipped)
  [mexwell             ]  17938 samples  (3459 skipped)
  [tushar5harma        ]  64727 samples  (2391 skipped)

Total: 132,171 samples | 64 unique class indices
Test set: 19,826 samples


In [15]:
# ── Eval Cell 3: Run inference ────────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import top_k_accuracy_score
from collections import Counter

all_preds  = []
all_labels = []
all_probs  = []

t0 = time.time()
with torch.no_grad():
    for i, (imgs, labels_batch) in enumerate(test_loader):
        imgs = imgs.to(device)
        logits = eval_model(imgs)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
        preds  = logits.argmax(dim=1).cpu().numpy()
        all_probs.append(probs)
        all_preds.append(preds)
        all_labels.append(labels_batch.numpy())
        if (i + 1) % 20 == 0:
            print(f"  {(i+1)*BATCH_SIZE} / {len(test_ds)}  ({time.time()-t0:.0f}s)")

all_preds  = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)
all_probs  = np.concatenate(all_probs, axis=0)
print("Inference complete.")

  2560 / 19826  (13s)
  5120 / 19826  (25s)
  7680 / 19826  (38s)
  10240 / 19826  (50s)
  12800 / 19826  (61s)
  15360 / 19826  (72s)
  17920 / 19826  (83s)
Inference complete.


In [16]:
# ── Eval Cell 4: Metrics ──────────────────────────────────────────────────────
present      = sorted(set(all_labels))
target_names = [ckpt_idx_to_class[i] for i in present]

top1 = (all_preds == all_labels).mean() * 100
top5 = top_k_accuracy_score(
    all_labels, all_probs, k=5,
    labels=list(range(ckpt_n_classes))
) * 100

print(f"Top-1 Accuracy : {top1:.4f} %")
print(f"Top-5 Accuracy : {top5:.4f} %")

print("\n── Per-class Report ─────────────────────────────────────────")
print(classification_report(
    all_labels, all_preds,
    labels=present,
    target_names=target_names,
    digits=4,
    zero_division=0,
))

Top-1 Accuracy : 87.9653 %
Top-5 Accuracy : 99.7579 %

── Per-class Report ─────────────────────────────────────────
                                precision    recall  f1-score   support

             Apple__Apple_Scab     0.9695    0.9259    0.9472       378
              Apple__Black_Rot     0.9296    0.9920    0.9598       373
       Apple__Cedar_Apple_Rust     1.0000    0.9576    0.9783       330
                Apple__Healthy     0.9472    0.9548    0.9510       376
   Bell_Pepper__Bacterial_Spot     0.9638    0.9638    0.9638       359
          Bell_Pepper__Healthy     0.9700    0.8660    0.9150       373
           Cashew__Anthracnose     0.8729    0.7881    0.8284       741
              Cashew__Gummosis     0.9431    0.9813    0.9618       321
               Cashew__Healthy     0.9218    0.8826    0.9018      1082
            Cashew__Leaf_Miner     0.8728    0.8869    0.8798       743
              Cashew__Red_Rust     0.9467    0.9563    0.9515       985
     Cassava__Bact

In [17]:
# ── Eval Cell 5: Per-class & per-crop breakdown ───────────────────────────────

# Per-class accuracy sorted worst → best
rows = []
for idx in present:
    mask      = all_labels == idx
    n_total   = mask.sum()
    n_correct = (all_preds[mask] == idx).sum()
    rows.append({
        "class"    : ckpt_idx_to_class[idx],
        "n"        : int(n_total),
        "correct"  : int(n_correct),
        "acc_%"    : round(n_correct / n_total * 100, 2) if n_total else 0.0,
    })

df_cls = pd.DataFrame(rows).sort_values("acc_%")
print("── 10 Worst Classes ─────────────────────────────────────────")
print(df_cls.head(10).to_string(index=False))
print("\n── 10 Best Classes ──────────────────────────────────────────")
print(df_cls.tail(10).to_string(index=False))

# Per-crop rollup
crop_prefixes = sorted({c.split("__")[0] for c in ckpt_class_to_idx})
crop_rows = []
for crop in crop_prefixes:
    idxs = [v for k, v in ckpt_class_to_idx.items() if k.startswith(crop + "__")]
    mask = np.isin(all_labels, idxs)
    if mask.sum() == 0:
        continue
    acc = (all_preds[mask] == all_labels[mask]).mean() * 100
    crop_rows.append({"crop": crop, "n": int(mask.sum()), "acc_%": round(acc, 2)})

df_crop = pd.DataFrame(crop_rows).sort_values("acc_%")
print("\n── Per-Crop Accuracy ────────────────────────────────────────")
print(df_crop.to_string(index=False))

── 10 Worst Classes ─────────────────────────────────────────
                        class    n  correct  acc_%
             Cassava__Healthy  325       84  25.85
      Coffee__Red_Spider_Mite   25       11  44.00
Cassava__Brown_Streak_Disease  275      135  49.09
        Cassava__Green_Mottle  299      152  50.84
    Cassava__Bacterial_Blight  138       72  52.17
                 Coffee__Rust   90       48  53.33
             Rice__Leaf_Blast  146       84  57.53
                Rice__Healthy  223      141  63.23
      Cassava__Mosaic_Disease 1654     1076  65.05
             Rice__Brown_Spot   92       65  70.65

── 10 Best Classes ──────────────────────────────────────────
                      class   n  correct  acc_%
         Wheat__Yellow_Rust 312      310  99.36
        Strawberry__Healthy 342      340  99.42
       Grape__Black_Measles 420      419  99.76
 Melon__Nitrogen_Deficiency   7        7 100.00
  Melon__Calcium_Deficiency   7        7 100.00
             Melon__Health